# Preamble

In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import polars as pl

In [ ]:
kagglehub.login()

In [ ]:
path = kagglehub.competition_download('store-sales-time-series-forecasting')

In [ ]:
type(path)
print("".join([path, '/train.csv']))

# Data Ingestion

In [ ]:
train_path = "".join([path, '/train.csv'])

In [ ]:
train = pd.read_csv("".join([path, '/train.csv']), parse_dates=['date'])
stores = pd.read_csv("".join([path, '/stores.csv']))

# Exploratory Data Analysis

## Dataset Overview

In [ ]:
print(train.head(5), "\n", train.tail(5))

In [ ]:
print(train.columns)

In [ ]:
train.describe()

In [ ]:
train.info()

In [ ]:
print(f"Number of categories or families and names: {train['family'].nunique()} \n {train['family'].unique()[:]}")


## Feature Engineering

In [ ]:
train['unique_id'] = train['store_nbr'].astype('str') + '_' + train['family']

In [ ]:
print(train.columns)
print(stores.columns)

In [ ]:
train.drop(columns='unique_id')

In [ ]:
print(f"Unique Time Series Count: {train['unique_id'].nunique()}")

In [ ]:
print(f"Date rage: {train['date'].min()} - {train['date'].max()}")

## Forecasting DataFrame

In [ ]:
df = train[["unique_id", "date", "sales"]].rename(columns={"date": "ds", "sales": "y"})

In [ ]:
df

In [ ]:
zero_pct = (df['y'] == 0).mean()
print(f"Zero sales percentage: {zero_pct:.1%}")

## Time Series Patterns

In [ ]:
# Plot a few series to see patterns
fix, axes = plt.subplots(10, 1, figsize=(14, 20))
sample_ids = df['unique_id'].sample(n=10, random_state=1)
for uid, ax in zip(sample_ids, axes):
    subset = df[df['unique_id'] == uid]
    ax.plot(subset['ds'], subset['y'])
    ax.set_title(uid)
plt.tight_layout()
plt.savefig('eda_sample_series.png')
plt.show()

In [ ]:
# Top 10 intermittent series
intermittent = df.groupby('unique_id')['y'].apply(lambda x: (x == 0).mean())
print(f"\n Top 10 most intermittent series:")
print(intermittent.sort_values(ascending=False).head(10))

# StatsForecast Baseline

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import (
    AutoARIMA,
    AutoETS,
    AutoCES,
    CrostonOptimized, # For intermittent demand
    SeasonalNaive, 
    HistoricAverage
)

## Data Subsetting and Train-Test Split

In [ ]:
print(df['unique_id'].nunique())

In [ ]:
# SUBSET first
subset_categories = ['1_BEVERAGES', '34_PREPARED FOODS', '8_CLEANING', '33_DAIRY', '10_GROCERY I']

In [ ]:
df_sample = df[df['unique_id'].isin(subset_categories)].copy()

In [ ]:
# Split: last 16 days as test (Kaggle uses 16-day horizon)
# df_sample['ds'] = pd.to_datetime(df_sample['ds']).dt.normalize()

max_date_per_series = df_sample.groupby('unique_id', as_index=False)['ds'].transform('max')

cutoff = max_date_per_series - pd.Timedelta(days=16)
train_df = df_sample[df_sample['ds'] <= cutoff]
test_df = df_sample[df_sample['ds'] > cutoff]

In [ ]:
print(train_df['ds'].max(), test_df['ds'].min())

In [ ]:
train_df.shape

In [ ]:
train_df.groupby('unique_id').mean()

In [ ]:
test_df.shape

## Model Training and Forecast

In [ ]:
# Define models

from statsforecast.utils import ConformalIntervals # This allow to calculate the Confidence intervals

intervals = ConformalIntervals(h=16, n_windows=2) # TODO: How this works.


models = [
    AutoARIMA(season_length=7, prediction_intervals=intervals),
    AutoETS(season_length=7, prediction_intervals=intervals),
    AutoCES(season_length=7, prediction_intervals=intervals),
    CrostonOptimized(prediction_intervals=intervals), # Intermittent demand
    SeasonalNaive(season_length=7, prediction_intervals=intervals), 
    HistoricAverage(prediction_intervals=intervals)
]

# Fit and forecast
sf = StatsForecast(
    models=models,
    freq='D', # Daily data
    n_jobs=-1, # Parallel
)


# Use .forecast() for speed (no fitted values stored)
forecasts = sf.forecast(df=train_df, h=16, level=[90])

In [ ]:
print(forecasts.head(6))

## Evaluation

In [ ]:
test_df.columns

In [ ]:
# Evaluation
eval_df = test_df.merge(forecasts.reset_index(), on=['unique_id', 'ds'])
model_cols = [c for c in forecasts.columns if not c in ['unique_id', 'ds'] and 'lo' not in c and 'hi' not in c]

In [ ]:
model_cols

In [ ]:
from  sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_log_error

# Evaluate each model for each series
mask = eval_df['unique_id'] == '8_CLEANING'
print(mean_absolute_error(eval_df.loc[mask, 'y'], eval_df.loc[mask, 'AutoARIMA']))

In [ ]:
# Function to evaluate each model for each series
def evaluate_models(eval_df: pd.DataFrame, model_cols: list) -> pd.DataFrame:
    results = []
    for uid in eval_df['unique_id'].unique():
        mask = eval_df['unique_id'] == uid
        for model in model_cols: 
            mae = mean_absolute_error(eval_df.loc[mask, 'y'], eval_df.loc[mask, model])
            rsme = np.sqrt(mean_squared_error(eval_df.loc[mask, 'y'], eval_df.loc[mask, model]))
            rmsle = root_mean_squared_log_error(eval_df.loc[mask, 'y'], eval_df.loc[mask, model])
            results.append({'unique_id': uid, 'model': model, 'MAE': mae, 'RMSE': rsme, 'RMSLE': rmsle})


    return pd.DataFrame(results)

def print_evaluation_results(results_df: pd.DataFrame):
    print("\n=== Model Comparison ===")
    print(results_df.pivot_table(index='model', values=['MAE', 'RMSE', 'RMSLE'], aggfunc='mean').round(2))

    best_mae = results_df.loc[results_df.groupby('unique_id')['MAE'].idxmin()]
    print("\n=== Best Model per Series (MAE) ===")
    print(best_mae[['unique_id', 'model', 'MAE']])

    best_rmse = results_df.loc[results_df.groupby('unique_id')['RMSE'].idxmin()]
    print("\n=== Best Model per Series (RMSE) ===")
    print(best_rmse[['unique_id', 'model', 'RMSE']])

    best_rmsle = results_df.loc[results_df.groupby('unique_id')['RMSLE'].idxmin()]
    print("\n=== Best Model per Series (RMSLE) ===")
    print(best_rmsle[['unique_id', 'model', 'RMSLE']])

In [ ]:
results_df = evaluate_models(eval_df, model_cols)
print_evaluation_results(results_df)

## Cross-Validation

In [ ]:
# StatForecast has built-in time series cross-validation

crossval = sf.cross_validation(
    df=df_sample,
    h=16,  # Forecast horizon 
    step_size=16, # Non-overlapping windows
    n_windows=3, # 3 folds
    level=[90], # Confidence intervals
)
print(crossval.head())

In [ ]:
print("\n=== MAE ===")
for model in model_cols:
    mae = mean_absolute_error(crossval['y'], crossval[model])
    print(f"{model}: CV MAE = {mae:.2f}")

print("\n=== RMSE ===")
for model in model_cols:
    rmse = np.sqrt(mean_squared_error(crossval['y'], crossval[model]))
    print(f"{model}: CV RMSE = {rmse:.2f}")

print("\n=== RMSLE ===")
for model in model_cols:
    rmsle = root_mean_squared_log_error(crossval['y'], crossval[model])
    print(f"{model}: CV RMSLE = {rmsle:.2f}")


## Forecast Visualization

In [ ]:
# Plot forecasts vs actual for 2-3 series
fig, axes = plt.subplots(len(subset_categories[:3]), 1, figsize=(14,10))
for ax, uid in zip(axes, subset_categories[:3]):
    # Historical
    hist = train_df[train_df['unique_id'] == uid].tail(60)
    ax.plot(hist['ds'], hist['y'], label='Historical', color='black')

    # Actuals in test period
    actual = test_df[test_df['unique_id'] == uid]
    actual = pd.concat([train_df[train_df['unique_id'] == uid].tail(1), actual])
    ax.plot(actual['ds'], actual['y'], label='Actual', color='green', linewidth=2)

    # Forecast (best 2 models)
    fc = forecasts.reset_index()
    fc_uid = fc[fc['unique_id'] == uid]
    ax.plot(fc_uid['ds'], fc_uid['AutoARIMA'], label='AutoARIMA', linestyle='--')
    ax.plot(fc_uid['ds'], fc_uid['AutoETS'], label='AutoETS', linestyle='--')

    # Confidence interval
    if 'AutoARIMA-lo-90' in fc_uid.columns:
        ax.fill_between(fc_uid['ds'], fc_uid['AutoARIMA-lo-90'],
                        fc_uid['AutoARIMA-hi-90'], alpha=0.2)
    ax.set_title(uid)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=150)
plt.show()

# Stretch Goals (if time allows)

- [x] Add oil.csv as exogenous variable to AutoARIMA
- [] Try NeuralForecast: NHITS, PatchTST on same data
- [ ] Add MLflow tracking for experiment logging
- [ ] Hierarchical forecasting: store-level → family-level reconciliation

# Exogenous Variables: Oil Price

## Data Loading

In [ ]:
oil_df = pd.read_csv(path + '/oil.csv', parse_dates=['date'])

In [ ]:
print(f"{oil_df.head(5)}", f"\n {oil_df.tail(5)}")

## Missing Values

In [ ]:
oil_df.isna().sum()

In [ ]:
oil_df = oil_df.ffill()


In [ ]:
oil_df.isna().sum()

In [ ]:
oil_df = oil_df.rename(columns={'dcoilwtico': 'oil_price'})

## Merge with Training and Test Data

In [ ]:
exo_train_df = train_df.copy().merge(oil_df, right_on='date', left_on='ds', how='left').drop(columns='date')

In [ ]:
exo_train_df.isna().sum()

In [ ]:
exo_test_df = test_df.copy().merge(oil_df, right_on='date', left_on='ds', how='left').drop(columns='date')

In [ ]:
exo_test_df.isna().sum()

In [ ]:
exo_test_df[exo_test_df['oil_price'].isna()]

In [ ]:
# Front and back fill with valid values after left join.
exo_test_df['oil_price'] = exo_test_df['oil_price'].ffill().bfill()
exo_train_df['oil_price'] = exo_train_df['oil_price'].ffill().bfill()

In [ ]:
exo_train_df.isna().sum()

In [ ]:
print(exo_test_df.shape, test_df.shape)
print(exo_test_df.columns)

## Model Training and Forecast

In [ ]:
# Define models

from statsforecast.utils import ConformalIntervals # This allow to calculate the Confidence intervals

intervals = ConformalIntervals(h=16, n_windows=2) # TODO: How this works.


models = [
    AutoARIMA(season_length=7, prediction_intervals=intervals),
    AutoETS(season_length=7, prediction_intervals=intervals),
    AutoCES(season_length=7, prediction_intervals=intervals),
    CrostonOptimized(prediction_intervals=intervals), # Intermittent demand
    SeasonalNaive(season_length=7, prediction_intervals=intervals), 
    HistoricAverage(prediction_intervals=intervals)
]

# Fit and forecast
sf = StatsForecast(
    models=models,
    freq='D', # Daily data
    n_jobs=-1, # Parallel
)


# Use .forecast() for speed (no fitted values stored)
forecasts = sf.forecast(df=exo_train_df, h=16, level=[90], X_df=exo_test_df[['unique_id', 'ds', 'oil_price']])

## Evaluation

In [ ]:
eval_df = exo_test_df.merge(forecasts.reset_index(), on=['unique_id', 'ds'])

In [ ]:
eval_df

In [ ]:
# Test evaluation for a specific series
mask = eval_df['unique_id'] == '8_CLEANING'
print(mean_absolute_error(eval_df.loc[mask, 'y'], eval_df.loc[mask, 'AutoARIMA']))

In [ ]:
evaluate_models(eval_df, model_cols)
print_evaluation_results(results_df)

## Cross-Validation with Exogenous Variables

In [ ]:
df_oily = df_sample.merge(oil_df, how='left', left_on='ds', right_on='date')

In [ ]:
df_oily = df_oily.drop(columns=['date'])

In [ ]:
df_oily = df_oily.ffill().bfill()

In [ ]:
# StatForecast has built-in time series cross-validation

crossval_exogen = sf.cross_validation(
    df=df_oily,
    h=16,
    step_size=16, # Non-overlapping windows
    n_windows=3, # 3 folds
    level=[90],
)
print(crossval_exogen.head())

In [ ]:
model_cols = [c for c in forecasts.columns if not c in ['unique_id', 'ds'] and 'lo' not in c and 'hi' not in c]

In [ ]:
print("\n=== MAE ===")

for model in model_cols:
    mae = mean_absolute_error(crossval_exogen['y'], crossval_exogen[model])
    print(f"{model}: CV MAE: {mae:.2f}")

print("\n=== RMSE ===")

for model in model_cols:
    rmse = np.sqrt(mean_squared_error(crossval_exogen['y'], crossval_exogen[model]))
    print(f"{model}: CV RMSE:{rmse:.2f}")

## Visualization

In [ ]:
fig, axes = plt.subplots(len(subset_categories[:3]), 1, figsize=(14,10))

for ax, uid in zip(axes, subset_categories[:3]):

    # Historical
    hist = exo_train_df[exo_train_df['unique_id'] == uid].tail(60)
    ax2 = ax.twinx()
    ax2.plot(hist['ds'], hist['oil_price'], label='Oil Price', color='black')
    ax.plot(hist['ds'], hist['y'], label='Historical', color='gold')


    # Actual in test period
    actual = exo_test_df[exo_test_df['unique_id'] == uid]
    ax.plot(actual['ds'], actual['y'], label='Actual',  color='green', linewidth=2)
    ax2.plot(actual['ds'], actual['oil_price'], label='Oil_price',  color='black', linewidth=2)
    
    
    ax.set_title(uid)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('forecast_with_oil_price_comparison.png', dpi=150)
plt.show()
    

# NeuralForecast

## Setup

In [ ]:
import logging
from neuralforecast import NeuralForecast
from neuralforecast.models import LSTM, GRU, RNN

In [ ]:
logging.getLogger('pytorch_lighting').setLevel(logging.ERROR)

## Model Training

In [ ]:
train_df.head(1)

In [ ]:
%%capture
horizon = 16

models = [
    LSTM(input_size=2 * horizon,
         h=horizon,
         max_steps=500,
         scaler_type='standard',
         encoder_hidden_size=64,
         decoder_hidden_size=64,
        ),
    
    GRU(input_size=2 * horizon,
         h=horizon,
         max_steps=500,
         scaler_type='standard',
         encoder_hidden_size=64,
         decoder_hidden_size=64,
        ),
    RNN(input_size=2 * horizon,
         h=horizon,
         max_steps=500,
         scaler_type='standard',
         encoder_hidden_size=64,
         decoder_hidden_size=64,
        ),
    
]

nf = NeuralForecast(models=models, freq='D')

nf.fit(df=train_df)



## Prediction and Evaluation

In [ ]:
y_hat = nf.predict(df=test_df)

In [ ]:
y_hat.head(10)

In [ ]:
fig, axes = plt.subplots(len(subset_categories[:3]), 1, figsize=(14,10))
for ax, uid in zip(axes, subset_categories[:3]):
    # Historical
    hist = train_df[train_df['unique_id'] == uid].tail(60)
    ax.plot(hist['ds'], hist['y'], label='Historical', color='black')

    # Actuals in test period
    actual = test_df[test_df['unique_id'] == uid]
    ax.plot(actual['ds'], actual['y'], label='Actual', color='green', linewidth=2)

    # Forecast (best 2 models)
    fc = y_hat.reset_index()
    fc_uid = fc[fc['unique_id'] == uid]
    ax.plot(fc_uid['ds'], fc_uid['LSTM'], label='LSTM', linestyle='--')
    ax.plot(fc_uid['ds'], fc_uid['RNN'], label='RNN', linestyle='--')
    ax.plot(fc_uid['ds'], fc_uid['GRU'], label='GRU', linestyle='--')

    # Confidence interval
    if 'AutoARIMA-lo-90' in fc_uid.columns:
        ax.fill_between(fc_uid['ds'], fc_uid['AutoARIMA-lo-90'],
                        fc_uid['AutoARIMA-hi-90'], alpha=0.2)
    ax.set_title(uid)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=150)
plt.show()

In [ ]:
eval_df = test_df.merge(y_hat.reset_index(), on=['unique_id', 'ds'])


In [ ]:

model_cols = [c for c in y_hat.columns if not c in ['unique_id', 'ds']]
results_df = evaluate_models(eval_df, model_cols)
print_evaluation_results(results_df)